# Daily History Backfill

Notebook version aligned with `backfill_daily.py`.
Use this for manual runs and debugging.

In [ ]:
import time
from datetime import date, datetime, timedelta, timezone

import pandas as pd
import pandas_market_calendars as mcal
import requests
from sqlalchemy import text

from backfill_utils import (
    ensure_schema,
    get_engine,
    insert_stock_ohlc,
    sleep_backoff,
    yahoo_symbol,
)

NYSE = mcal.get_calendar("NYSE")

In [ ]:
def last_trading_schedule(n_sessions: int, lookback_days: int = 365) -> pd.DataFrame:
    end = pd.Timestamp.now("UTC")
    start = end - pd.Timedelta(days=lookback_days)
    sched = NYSE.schedule(start_date=start, end_date=end)
    return sched.tail(n_sessions)


def fetch_daily_yahoo_chart(symbol: str, start_date: date, end_date: date):
    sym_fetch = yahoo_symbol(symbol)

    start_dt = datetime(start_date.year, start_date.month, start_date.day, tzinfo=timezone.utc)
    end_dt = datetime(end_date.year, end_date.month, end_date.day, tzinfo=timezone.utc) + timedelta(days=1)

    period1 = int(start_dt.timestamp())
    period2 = int(end_dt.timestamp())

    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{sym_fetch}"
    params = {
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
    }
    headers = {"User-Agent": "Mozilla/5.0"}

    resp = requests.get(url, params=params, headers=headers, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    if not data.get("chart") or not data["chart"].get("result"):
        return None

    result = data["chart"]["result"][0]
    timestamps = result.get("timestamp", [])
    if not timestamps:
        return None

    quote = result["indicators"]["quote"][0]
    ts = pd.Series(timestamps, dtype="int64")

    df = pd.DataFrame(
        {
            "symbol": symbol,
            "data_type": "1d",
            "ts": ts,
            "open": quote.get("open"),
            "high": quote.get("high"),
            "low": quote.get("low"),
            "close": quote.get("close"),
            "volume": quote.get("volume"),
        }
    ).dropna(subset=["open", "high", "low", "close"])

    if df.empty:
        return None

    return df[["symbol", "data_type", "ts", "open", "high", "low", "close", "volume"]]


def load_active_symbols_from_db(engine) -> list[str]:
    with engine.begin() as conn:
        rows = conn.execute(
            text(
                """
SELECT symbol
FROM stocks
WHERE status = true
ORDER BY symbol
"""
            )
        ).fetchall()
    return [r[0] for r in rows]


In [ ]:
def run_daily_history(
    symbols: list[str] | None = None,
    mode: str = "missing",
    start_date_str: str | None = None,
    sessions: int = 180,
    max_attempts: int = 3,
    sleep_sec: float = 0.0,
):
    engine = get_engine()
    ensure_schema(engine)
    total = 0

    if not symbols:
        symbols = load_active_symbols_from_db(engine)

    if not symbols:
        engine.dispose()
        raise ValueError("No symbols provided and no active symbols found in DB.")

    if mode == "initial":
        start_date = date.fromisoformat(start_date_str) if start_date_str else date(2015, 1, 1)
        end_date = date.today() - timedelta(days=1)
        per_symbol_ranges = [(start_date, end_date)]
    else:
        sched = last_trading_schedule(sessions)
        start_date = sched.index[0].date()
        end_date = sched.index[-1].date()
        per_symbol_ranges = [(start_date, end_date)]

    for sym in symbols:
        sym = sym.strip()
        if not sym:
            continue

        for start_d, end_d in per_symbol_ranges:
            for attempt in range(max_attempts):
                try:
                    df = fetch_daily_yahoo_chart(sym, start_d, end_d)
                    with engine.begin() as conn:
                        total += insert_stock_ohlc(conn, df)
                    break
                except Exception as e:
                    print(f"[daily {mode}] {sym} {start_d}..{end_d} attempt={attempt+1}/{max_attempts} error={e}")
                    if attempt < max_attempts - 1:
                        sleep_backoff(attempt)
                    else:
                        print(f"[daily {mode}] {sym} give up for now")

        if sleep_sec:
            time.sleep(float(sleep_sec))

    engine.dispose()
    print(f"[daily {mode}] Total inserted: {total}")

In [ ]:
# Examples:
# run_daily_history(mode="initial", start_date_str="2015-01-01")
# run_daily_history(mode="missing", sessions=180)

run_daily_history(mode="missing", sessions=180)